In [1]:
import json

# Only works for some .json exported files from nsys-ui...
def process_json_report(filePath):
    kernel_trace = []
    kernel_names = []
    
    with open(filePath, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                if entry.get('type') == 79:
                    kernel_trace.append(line.get('CudaEvent'))
                    
                if entry.get('type') == 'String':
                     kernel_names.append(entry['value'])
            except json.JSONDecodeError as e:
                print(f"Skipping malformed line: {e}")
    return (kernel_trace, kernel_names)

In [2]:
import sqlite3

def process_sqlite_report(filePath):
        
    con = sqlite3.connect(filePath)
    con.row_factory = sqlite3.Row 
    cur = con.cursor()
    
    query = """
    SELECT
        names.value AS kernel_name,
        k.start,
        k.end,
        k.end - k.start AS duration,
        k.registersPerThread,
        k.gridX, k.gridY, k.gridZ,
        k.blockX, k.blockY, k.blockZ,
        k.staticSharedMemory, k.dynamicSharedMemory, k.localMemoryPerThread, k.localMemoryTotal, k.sharedMemoryLimitConfig,
        k.streamId,
        k.deviceId
    FROM CUPTI_ACTIVITY_KIND_KERNEL AS k
    JOIN StringIds AS names ON k.demangledName = names.id
    ORDER BY k.start;
    """
    cur.execute(query)
    
    rows = [dict(row) for row in cur.fetchall()]

    return rows
    

In [3]:
rows = process_sqlite_report("build/Linux-x86_64/report101_1575MHz_bindless_normal.sqlite")
print(rows[0])
print(rows[1])

{'kernel_name': 'Typeinfo name for popsift::normalizedSource::Horiz<(bool)0>', 'start': 313482745, 'end': 313878203, 'duration': 395458, 'registersPerThread': 28, 'gridX': 30, 'gridY': 2160, 'gridZ': 1, 'blockX': 128, 'blockY': 1, 'blockZ': 1, 'staticSharedMemory': 0, 'dynamicSharedMemory': 0, 'localMemoryPerThread': 0, 'localMemoryTotal': 42467328, 'sharedMemoryLimitConfig': 1, 'streamId': 39, 'deviceId': 0}
{'kernel_name': 'Typeinfo name for popsift::vert_kernel', 'start': 313885563, 'end': 314132477, 'duration': 246914, 'registersPerThread': 30, 'gridX': 60, 'gridY': 1080, 'gridZ': 1, 'blockX': 64, 'blockY': 2, 'blockZ': 1, 'staticSharedMemory': 0, 'dynamicSharedMemory': 0, 'localMemoryPerThread': 0, 'localMemoryTotal': 42467328, 'sharedMemoryLimitConfig': 1, 'streamId': 39, 'deviceId': 0}


In [4]:
rows[110]["kernel_name"]

'Typeinfo name for popsift::find_extrema_in_dog<(int)4, (int)1>'

In [5]:
import re
        # match = re.search(r'popsift::[a-zA-Z_][a-zA-Z_0-9]*(?:<[^>]+>)?', row["kernel_name"])
def clean_kernel_name(data):
    unique_names = []
    stufus = []
    it = 0
    for row in data:
        match = re.search(r'Normalize|popsift::[a-zA-Z_][a-zA-Z_0-9]*', row["kernel_name"])
        match_template = re.search(r'Normalize|popsift::\w+(::\w+)*<[^>]+>$', row["kernel_name"])
        
        if match_template:
            if match_template.group() not in unique_names:
                unique_names.append(match_template.group())
            row["kernel_name"] = match_template.group() # Update with shorter name
        elif match:
            if match.group() not in unique_names:
                unique_names.append(match.group())
            row["kernel_name"] = match.group() # Update with shorter name

        else:
            print("Well shit ", row["kernel_name"])

    return unique_names

names = clean_kernel_name(rows)

In [6]:
names

['popsift::normalizedSource::Horiz<(bool)0>',
 'popsift::vert_kernel',
 'popsift::absoluteSource::Horiz<(bool)1, (bool)0>',
 'popsift::Downscale',
 'popsift::make_dog',
 'popsift::find_extrema_in_dog<(int)4, (int)1>',
 'popsift::ori_par_subgroup',
 'popsift::ori_prefix_sum_subgroup',
 'popsift::sub_group_desc_loop',
 'Normalize',
 'popsift::Prep_features']

In [7]:
print(rows[110]["kernel_name"])

popsift::find_extrema_in_dog<(int)4, (int)1>


In [8]:
import pandas as pd

df = pd.read_csv("build/Linux-x86_64/metadata_report101_1575MHz_bindless_normal.csv")
print(df.iloc[0].loc['filename'])

0001.png


In [50]:
def frame_time_kernel(data, k_names, warmup_frame_count):
    # They are in order of occurance 
    in_frame = False
    start_time = 0
    frame_times = []
    frame_count = 0
    for row in data:
        if not in_frame and row["kernel_name"] == k_names[0]:
            start_time = row["start"]
            in_frame = True
            
        if in_frame and row["kernel_name"] == k_names[-1]:
            if frame_count >= warmup_frame_count:
                # First 10 frames are warmup frames (in my initial case)
                frame_times.append(row["end"] - start_time)
                in_frame = False
            else:
                frame_count += 1
            
            
    return frame_times
    # print(data[0]["kernel_name"])     

In [51]:
times = frame_time_kernel(rows, names, 10)

In [52]:
len(times)

12000

In [29]:
df.iloc[799]

filename            0800.png
feature_count           9874
descriptor_count       11694
Name: 799, dtype: object

In [32]:
len(df)

12000